# Dealing with Lookahead Conflicts

This notebook discusses conflicts that have their origin in insufficient lookahead.
We will discuss the following grammar:
```
    a : b "U" "V"
      | c "U" "W"

    b : "X"

    c : "X"
```
This grammar is *unambiguous*:  There are only two string that can be derived from the start symbol `a`:
 * `"XUV"`,
 * `"XUW"`.

Each of these strings has exactly one parse tree.
Nevertheless, the grammar does not have the `LR(1)` property and is therefore certainly not an `LALR(1)` grammar.

## Specification of the Grammar

Below is the grammar from above written as a scanner specification for `ocamllex` and a grammar for `Menhir`.  (The
Python version of this notebook uses `Lark`.)  The tokens `"U"`, `"V"`, `"W"`, and `"X"` carry a value of type
`Parse_tree.tok`, so that they can be kept in the parse tree.  As usual, `Menhir` needs a start rule that is followed
by the end of the input `EOF`.

In [ ]:
let lexer = {|
{
  open A_parser
}

rule token = parse
  | 'U'                          { U (Parse_tree.token "U" lexbuf) }
  | 'V'                          { V (Parse_tree.token "V" lexbuf) }
  | 'W'                          { W (Parse_tree.token "W" lexbuf) }
  | 'X'                          { X (Parse_tree.token "X" lexbuf) }
  | eof                          { EOF }
  | _                            { raise (Parse_tree.unexpected lexbuf ("No terminal matches " ^ repr (Lexing.lexeme lexbuf))) }
|}

let grammar = {|
%{
  open Parse_tree
%}

%token <Parse_tree.tok> U V W X
%token EOF

%start <Parse_tree.tree> start
%type  <Parse_tree.tree> a b c

%%

start: x = a EOF               { x }

a : x = b u = U v = V          { Tree ("a", [x; Token u; Token v]) }
  | x = c u = U w = W          { Tree ("a", [x; Token u; Token w]) }

b : x = X                      { Tree ("b", [Token x]) }

c : x = X                      { Tree ("c", [Token x]) }
|}

We load the file `../lib/parser_generator.ml`, which provides the functions `generate_parser` and
`generate_parser_with`.  `Menhir` always prints its warnings, so there is no logger to configure.

In [ ]:
#use "../lib/nb.ml";;
load_lib "parser_generator.ml";;
open Parse_tree

## Trying to Build an `LALR` Parser

`Lark` always raises an exception of class `GrammarError` when it tries to build a parser for a grammar that contains a
*reduce/reduce* conflict.  `Menhir` is more lenient:  it only prints a warning and resolves a *reduce/reduce* conflict
in favour of the rule that occurs first in the grammar.  This would mean that the string `XUW` could never be
parsed.  Therefore, we call `Menhir` with the option `--strict`, which turns every warning into an error.  Then
`generate_parser_with` raises the exception `Grammar_error`.

In order to show the offending state, we print the file `_build/a_parser.conflicts`, which `Menhir` writes because of
the option `--explain`.

In [ ]:
try
  generate_parser_with ~options:"--lalr --strict" ~name:"a" ~lexer ~grammar;
  print_endline "No conflict."
with Grammar_error e ->
  print_string e;
  cat "_build/a_parser.conflicts"

## Why Is There a Conflict?

If we compute the `LR` states of this grammar, we find, among others, the state
$$ \{\; b \rightarrow \texttt{'X'} \bullet : \texttt{'U'}, \quad
       c \rightarrow \texttt{'X'} \bullet : \texttt{'U'} \;\}, $$
which is exactly the state shown in the explanation above:  it is the state that is reached after reading `X`.
Since the set of follow tokens is the same for both rules, we have a *reduce/reduce* conflict here.

The conflict is caused by the fact that the parser, with a lookahead of only *one* token, cannot decide
whether an `'X'` should be interpreted as a `b` or as a `c`, because this is only decided when the character
*following* the `'U'` is read:

* if this is a `'V'`, then the rule $a \rightarrow b\,\texttt{'U'}\,\texttt{'V'}$ is used and consequently
  the `'X'` has to be interpreted as a `b`;
* if, however, the second token after the `'X'` is a `'W'`, then the rule
  $a \rightarrow c\,\texttt{'U'}\,\texttt{'W'}$ is used and consequently the `'X'` is to be read as a `c`.

Since `Menhir` has refused to create a parser, there is no parser that we could use.  The file
`_build/a_parser.automaton` contains the states of the automaton nevertheless.

## A Way Out: A `GLR` Parser

`Lark` offers an escape hatch that the classical parser generators of the `yacc` family do not have: the *default* parser
of `Lark` is an `Earley` parser, and the `Earley` algorithm does not need a bounded lookahead at all.  `Menhir` offers a
similar escape hatch:  with the option `--GLR`, it creates a *generalized LR parser*.  In a state with a conflict, a
`GLR` parser pursues all possible actions in parallel.  Therefore, the grammar is accepted as it stands.  A `GLR` parser
needs the library `menhirGLR`, which `generate_parser_with` loads.

`Menhir` prints a warning that a *merge function* should be provided.  A merge function decides which parse tree is
returned if a string has more than one parse tree.  Since our grammar is unambiguous, we need no merge function.

The `Lark` version needs the keyword argument `keep_all_tokens=True` in order to keep the anonymous terminals in the
parse tree.  In `Menhir`, the semantic actions given above put the tokens into the tree.

The price for this convenience is a parser that is considerably slower than an `LALR` parser.

In [ ]:
generate_parser_with ~options:"--GLR" ~name:"a" ~lexer ~grammar

The `GLR` parser has no difficulty deciding whether the `'X'` is a `b` or a `c`:  it simply pursues both
possibilities until the last token settles the question.

In [ ]:
print_string (pretty (parse_string A_parser.start A_lexer.token "XUV"));;
flush stdout

In [ ]:
print_string (pretty (parse_string A_parser.start A_lexer.token "XUW"));;
flush stdout

## The Real Solution

Switching to a `GLR` parser hides the problem instead of solving it.  The grammar can be rewritten so
that it becomes an `LALR(1)` grammar:  if the context `'U'`, which follows both `b` and `c`, is pulled into
the rules for `b` and `c`, then the conflict disappears, because the state in which the conflict occurred now
has the form
$$ \{\; b \rightarrow \texttt{'X'}\,\texttt{'U'} \bullet : \texttt{'V'}, \quad
       c \rightarrow \texttt{'X'}\,\texttt{'U'} \bullet : \texttt{'W'} \;\} $$
and here the next token decides which rule to reduce with.  This is done in the notebook
[04-Look-Ahead-Solved.ipynb](04-Look-Ahead-Solved.ipynb).